# Step 3: masked pooling
`MaskedPooling` turns the BiLSTM output `(B, T, 2·hidden)` into one vector per address `(B, 2·hidden)`, ignoring pad positions. Built and tested here on hand-made `h` and `lengths`, so no vocab, embedding or LSTM is involved.
The class itself lives in `08_encoder_step.py`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import project modules

import torch
from torch import nn

In [2]:
# the class lives in 08_encoder_step.py (single copy, shared with training); this notebook tests it
MaskedPooling = import_module("siamese_model.08_encoder_step").MaskedPooling

## Tests

In [3]:
torch.manual_seed(0)
x = torch.tensor([[[1.], [2.], [3.]]])                 # B=1, T=3, D=1
mul_x = torch.tensor([[[1., 9.], [7., 2.], [4., 4.]]])  # D=2: per-feature maxima sit at different positions
length_1 = torch.tensor([3])                            # long, like encode_batch's lengths
length_2 = torch.tensor([2])                            # last position becomes a pad

# mean: pads are excluded from the sum and the count
pooling = MaskedPooling(mode="mean")
with torch.no_grad():
    assert pooling(x, length_1).item() == 2
    assert pooling(x, length_2).item() == 1.5           # the pad (3) would make it 2 if it leaked in
    assert torch.allclose(pooling(mul_x, length_1), torch.tensor([[4., 5.]]))

# max: pads can never win
pooling = MaskedPooling(mode="max")
with torch.no_grad():
    assert pooling(x, length_1).item() == 3
    assert pooling(x, length_2).item() == 2             # the pad (3) would win if it leaked in
    assert torch.equal(pooling(mul_x, length_1), torch.tensor([[7., 9.]]))   # max is per feature, not one best position

# max with all-negative real values and a 0 pad (what pad_packed_sequence produces)
neg_x = torch.tensor([[[-3.], [-1.], [0.]]])
with torch.no_grad():
    assert pooling(neg_x, length_1).item() == 0         # control: length 3, so the 0 is real here
    assert pooling(neg_x, length_2).item() == -1        # 0 is a pad: without the -inf fill the answer would be 0
print("by-hand values: passed")

by-hand values: passed


In [4]:
# shapes and parameters on a realistic batch
x = torch.randn(3, 4, 128)
lengths = torch.tensor([1, 3, 2])
mask = torch.arange(4)[None, :] < lengths[:, None]     # (3, 4), True = real position
mean_pooling, max_pooling = MaskedPooling(mode="mean"), MaskedPooling(mode="max")
assert mean_pooling(x, lengths).shape == (3, 128)
assert max_pooling(x, lengths).shape == (3, 128)
assert sum(p.numel() for p in mean_pooling.parameters()) == 0   # no layers -> nothing to train

# padding invariance: garbage in every pad position changes nothing,
# both inside the original width (rows of different lengths) and beyond it (extra padding)
x_garbage = x.clone()
x_garbage[~mask] = 100.0
x_garbage = torch.cat([x_garbage, torch.randn(3, 4, 128) * 100], dim=1)   # T = 4 -> 8
for pool in (mean_pooling, max_pooling):
    assert torch.allclose(pool(x, lengths), pool(x_garbage, lengths), atol=1e-6)
print("shapes / invariance: passed")

shapes / invariance: passed


In [5]:
# gradient flows back to the real positions (so the LSTM can learn) and never to the pads
lengths = torch.tensor([1, 3, 2])
mask = torch.arange(4)[None, :] < lengths[:, None]     # (3, 4)

x = torch.randn(3, 4, 1, requires_grad=True)          # randn, not ones: ties would muddy the max case
MaskedPooling(mode="mean")(x, lengths).sum().backward()
assert (x.grad[~mask] == 0).all()                                              # pads get nothing
assert torch.allclose(x.grad[mask].squeeze(-1), (1 / lengths.float()).repeat_interleave(lengths))   # each real position gets 1/length

x = torch.randn(3, 4, 1, requires_grad=True)          # fresh tensor: gradients would otherwise accumulate
MaskedPooling(mode="max")(x, lengths).sum().backward()
assert (x.grad[~mask] == 0).all()                                              # pads get nothing
assert torch.equal(x.grad.sum(1), torch.ones(3, 1))                           # exactly one winning position per row and feature
print("gradients: passed")

gradients: passed


In [6]:
# unknown mode is rejected (ValueError only, so a typo elsewhere can't make this pass)
try:
    MaskedPooling(mode="garbage")
except ValueError:
    pass
else:
    assert False, "MaskedPooling accepted an unknown mode"
print("bad mode: passed")

bad mode: passed
